# Downstream benchmark with configurable neural probes

This notebook follows the subject-wise evaluation used in `linear_probing.ipynb`, but trains the neural head selected by `head` in `utils/configs/<model>.yaml` instead of a scikit-learn linear classifier. Models, datasets, and channel modes (`mapped`, `raw`, or both) are selected with lists in the final cell.

In [14]:
import copy
import os
import random
import sys
from pathlib import Path

os.environ.setdefault("MNE_DONTWRITE_HOME", "true")
os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yaml
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from torch.utils.data import DataLoader

ROOT = Path.cwd()
if not (ROOT / "utils" / "configs").is_dir():
    ROOT = ROOT / "benchmark"
if not (ROOT / "utils" / "configs").is_dir():
    raise FileNotFoundError("Run this notebook from the repository root or benchmark/.")
sys.path.insert(0, str(ROOT))

from eeg_preprocessing.loaders import CustomDataset, get_subject_ids

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LMDB_PATHS = {
    "BCI2A": "/projects/EEG-foundation-model/RECH202/data_preprocessed_normalized/BCI2A/bci2a.lmdb",
    "FACED": "/projects/EEG-foundation-model/RECH202/data_preprocessed_normalized/FACED/faced.lmdb",
    "PHYSIONETMI": "/projects/EEG-foundation-model/RECH202/data_preprocessed_normalized/PHYSIONETMI/physionetmi.lmdb",
}

with open(ROOT / "utils" / "downstream_data.yaml") as file:
    DOWNSTREAM_CONFIG = yaml.safe_load(file)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [15]:
def load_model_config(model_name):
    config_path = ROOT / "utils" / "configs" / f"{model_name}.yaml"
    with open(config_path) as file:
        return yaml.safe_load(file)


def load_wrapper_class(model_name):
    if model_name == "cbramod":
        from utils.wrappers.cbramod import CBraModWrapper
        return CBraModWrapper
    if model_name == "csbrain":
        from utils.wrappers.csbrain import CSBrainWrapper
        return CSBrainWrapper
    if model_name == "eegfm":
        from utils.wrappers.eegfm import EEGFMWrapper
        return EEGFMWrapper
    if model_name == "eegmamba":
        from utils.wrappers.eegmamba import EEGMambaWrapper
        return EEGMambaWrapper
    if model_name == "eegnet":
        from utils.wrappers.eegnet import EEGNetWrapper
        return EEGNetWrapper
    if model_name == "eegpt":
        from utils.wrappers.eegpt import EEGPTWrapper
        return EEGPTWrapper
    if model_name == "labram":
        from utils.wrappers.labram import LaBraMWrapper
        return LaBraMWrapper
    if model_name == "lumamba":
        from utils.wrappers.lumamba import LuMambaWrapper
        return LuMambaWrapper
    if model_name == "luna":
        from utils.wrappers.luna import LUNAWrapper
        return LUNAWrapper
    if model_name == "reve":
        from utils.wrappers.reve import ReveWrapper
        return ReveWrapper
    raise ValueError(f"Unknown model: {model_name}")


class ProbeModel(nn.Module):
    def __init__(self, wrapper, frozen):
        super().__init__()
        self.wrapper = wrapper
        self.frozen = frozen
        self.backbone = wrapper.backbone
        self.classifier = wrapper.classifier
        if hasattr(wrapper, "chan_conv"):
            self.chan_conv = wrapper.chan_conv

    def train(self, mode=True):
        super().train(mode)
        if self.frozen:
            self.backbone.eval()
        return self

    def forward(self, x):
        return self.wrapper.forward(self.wrapper.preprocess(x))


def create_probe_model(model_name, dataset_name, channel_mode=None, device=DEVICE):
    config = load_model_config(model_name)
    if model_name != "eegnet" and "head" not in config:
        raise KeyError(f"Missing 'head' in utils/configs/{model_name}.yaml")

    channel_mode = channel_mode or config.get("channel_mode", "mapped")
    if channel_mode not in {"mapped", "raw"}:
        raise ValueError(f"Unknown channel mode: {channel_mode!r}")
    lmdb_path = LMDB_PATHS[dataset_name]
    first_subject = get_subject_ids(Path(lmdb_path))[0]
    sample_dataset = CustomDataset(
        lmdb_path,
        subject_ids=[first_subject],
        channel_mode=channel_mode,
        return_ch_names=channel_mode == "raw",
    )
    sample = sample_dataset[0]
    x_sample = sample[0]
    ch_names = sample[2] if channel_mode == "raw" else None

    checkpoint = None
    if config.get("use_pretrained_weights", False):
        checkpoint = ROOT / config["pretrained_weights"]
        if not checkpoint.is_file():
            raise FileNotFoundError(f"Missing pretrained checkpoint: {checkpoint}")

    wrapper = load_wrapper_class(model_name)(
        ckpt_path=str(checkpoint) if checkpoint is not None else None,
        dropout=config.get("dropout", 0.0),
    )
    wrapper.build_backbone(
        n_channels=x_sample.shape[0],
        time_length=x_sample.shape[1],
        ch_names=ch_names,
    )

    head_kwargs = {
        key: config[key]
        for key in ("head", "dropout", "attn_heads", "attn_queries",
                    "hidden_dim", "probe_max_norm", "classifier_max_norm")
        if key in config
    }
    n_classes = DOWNSTREAM_CONFIG[dataset_name]["n_classes"]
    if n_classes is None:
        raise ValueError(f"n_classes is not configured for {dataset_name}")
    wrapper.build_classifier(n_classes=n_classes, **head_kwargs)

    frozen = config.get("frozen", True)
    if frozen:
        wrapper.freeze_backbone()
    model = ProbeModel(wrapper, frozen=frozen).to(device)
    return model, config, channel_mode


In [16]:
def split_subjects(subject_ids, split_ratio, seed):
    subject_ids = np.asarray(subject_ids, dtype=object)
    if len(subject_ids) < 3:
        raise ValueError("A train/validation/test subject split requires at least 3 subjects.")
    ratios = np.asarray(split_ratio, dtype=float)
    ratios = ratios / ratios.sum()
    shuffled = np.random.default_rng(seed).permutation(subject_ids)
    n_test = max(1, int(round(len(shuffled) * ratios[2])))
    n_val = max(1, int(round(len(shuffled) * ratios[1])))
    while n_test + n_val >= len(shuffled):
        if n_test >= n_val and n_test > 1:
            n_test -= 1
        elif n_val > 1:
            n_val -= 1
        else:
            raise ValueError("Not enough subjects to create three non-empty splits.")
    test_subjects = shuffled[:n_test].tolist()
    val_subjects = shuffled[n_test:n_test + n_val].tolist()
    train_subjects = shuffled[n_test + n_val:].tolist()
    return train_subjects, val_subjects, test_subjects


def make_dataset(dataset_name, subject_ids, channel_mode):
    data_config = DOWNSTREAM_CONFIG["data"]
    return CustomDataset(
        LMDB_PATHS[dataset_name],
        subject_ids=subject_ids,
        channel_mode=channel_mode,
        return_ch_names=False,
        scale_factor=data_config.get("scale_factor", 1.0),
    )


def make_loader(dataset, batch_size, shuffle, seed):
    data_config = DOWNSTREAM_CONFIG["data"]
    num_workers = data_config.get("num_workers", 0)
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=data_config.get("pin_memory", torch.cuda.is_available()),
        persistent_workers=data_config.get("persistent_workers", False) and num_workers > 0,
        generator=generator,
    )


def create_dataloaders(dataset_name, channel_mode, batch_size, seed):
    subjects = list(get_subject_ids(Path(LMDB_PATHS[dataset_name])))
    train_subjects, val_subjects, test_subjects = split_subjects(
        subjects, DOWNSTREAM_CONFIG["data"]["split_ratio"], seed
    )
    loaders = {
        "train": make_loader(
            make_dataset(dataset_name, train_subjects, channel_mode), batch_size, True, seed
        ),
        "val": make_loader(
            make_dataset(dataset_name, val_subjects, channel_mode), batch_size, False, seed
        ),
        "test": make_loader(
            make_dataset(dataset_name, test_subjects, channel_mode), batch_size, False, seed
        ),
    }
    splits = {"train": train_subjects, "val": val_subjects, "test": test_subjects}
    return loaders, splits


In [17]:
def classification_metrics(labels, predictions):
    return {
        "accuracy": accuracy_score(labels, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro", zero_division=0),
    }


def run_epoch(model, loader, criterion, optimizer=None, clip_value=None, device=DEVICE):
    training = optimizer is not None
    model.train(training)
    losses, labels, predictions = [], [], []
    # Some probe heads enforce max-norm constraints by updating weights in forward.
    # no_grad keeps validation safe without turning those weights into inference tensors.
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for batch in loader:
            x, y = batch[:2]
            x = x.to(device, dtype=torch.float32, non_blocking=True)
            y = y.to(device, dtype=torch.long, non_blocking=True).view(-1)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            if training:
                loss.backward()
                if clip_value is not None:
                    nn.utils.clip_grad_norm_(
                        (parameter for parameter in model.parameters() if parameter.requires_grad),
                        clip_value,
                    )
                optimizer.step()
            losses.append(loss.detach().item() * len(y))
            labels.append(y.detach().cpu().numpy())
            predictions.append(logits.detach().argmax(dim=1).cpu().numpy())
    labels = np.concatenate(labels)
    predictions = np.concatenate(predictions)
    metrics = classification_metrics(labels, predictions)
    metrics["loss"] = sum(losses) / len(labels)
    return metrics


def create_optimizer(model, config):
    optimizer_class = getattr(torch.optim, config.get("optimizer", "AdamW"))
    lr = config.get("lr", 4e-4)
    weight_decay = config.get("weight_decay", 0.0)
    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if not trainable:
        raise ValueError("The selected configuration has no trainable parameters.")

    if config.get("multi_lr", False) and not model.frozen:
        head_parameters = list(model.classifier.parameters())
        if hasattr(model, "chan_conv"):
            head_parameters += list(model.chan_conv.parameters())
        head_ids = {id(parameter) for parameter in head_parameters}
        body_parameters = [parameter for parameter in trainable if id(parameter) not in head_ids]
        parameter_groups = [{"params": head_parameters, "lr": lr}]
        if body_parameters:
            parameter_groups.append({"params": body_parameters, "lr": lr * 0.1})
    else:
        parameter_groups = trainable
    return optimizer_class(parameter_groups, lr=lr, weight_decay=weight_decay)


def snapshot_trainable_modules(model):
    state = {"classifier": copy.deepcopy(model.classifier.state_dict())}
    if hasattr(model, "chan_conv"):
        state["chan_conv"] = copy.deepcopy(model.chan_conv.state_dict())
    if not model.frozen:
        state["backbone"] = copy.deepcopy(model.backbone.state_dict())
    return state


def restore_trainable_modules(model, state):
    model.classifier.load_state_dict(state["classifier"])
    if "chan_conv" in state:
        model.chan_conv.load_state_dict(state["chan_conv"])
    if "backbone" in state:
        model.backbone.load_state_dict(state["backbone"])


def fit_probe(model, loaders, config, device=DEVICE):
    criterion = nn.CrossEntropyLoss(label_smoothing=config.get("label_smoothing", 0.0))
    optimizer = create_optimizer(model, config)
    best_score = -np.inf
    best_epoch = 0
    best_state = None
    patience = config.get("early_stopping_patience", 10)
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, config.get("epochs", 50) + 1):
        train_metrics = run_epoch(
            model, loaders["train"], criterion, optimizer, config.get("clip_value"), device
        )
        val_metrics = run_epoch(model, loaders["val"], criterion, device=device)
        row = {"epoch": epoch}
        row.update({f"train_{key}": value for key, value in train_metrics.items()})
        row.update({f"val_{key}": value for key, value in val_metrics.items()})
        history.append(row)
        if val_metrics["balanced_accuracy"] > best_score:
            best_score = val_metrics["balanced_accuracy"]
            best_epoch = epoch
            best_state = snapshot_trainable_modules(model)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        print(
            f"Epoch {epoch:03d}/{config.get('epochs', 50):03d} | "
            f"train loss={train_metrics['loss']:.4f}, bacc={train_metrics['balanced_accuracy']:.4f} | "
            f"val loss={val_metrics['loss']:.4f}, bacc={val_metrics['balanced_accuracy']:.4f}"
        )
        if epochs_without_improvement >= patience:
            print(
                f"Early stopping after {epoch} epochs: validation balanced accuracy "
                f"did not improve for {patience} consecutive epochs."
            )
            break

    restore_trainable_modules(model, best_state)
    test_metrics = run_epoch(model, loaders["test"], criterion, device=device)
    return pd.DataFrame(history), test_metrics, best_epoch


In [18]:
def train_probe(model_name, dataset_name, channel_mode=None, device=DEVICE):
    config = load_model_config(model_name)
    seed = config.get("data_seed", config.get("seed", 100))
    seed_everything(seed)
    channel_mode = channel_mode or config.get("channel_mode", "mapped")
    if channel_mode not in {"mapped", "raw"}:
        raise ValueError(f"Unknown channel mode: {channel_mode!r}")
    loaders, splits = create_dataloaders(
        dataset_name, channel_mode, config.get("batch_size", 64), seed
    )
    model, config, channel_mode = create_probe_model(
        model_name, dataset_name, channel_mode, device
    )

    print(
        f"Model={model_name}, head={config.get('head', 'official')}, dataset={dataset_name}, "
        f"channel_mode={channel_mode}, device={device}"
    )
    print(
        f"Subjects: train={len(splits['train'])}, val={len(splits['val'])}, "
        f"test={len(splits['test'])}"
    )
    model.wrapper.summary()
    history, test_metrics, best_epoch = fit_probe(model, loaders, config, device)
    result = {
        "model": model_name,
        "head": config.get("head", "official"),
        "dataset": dataset_name,
        "channel_mode": channel_mode,
        "frozen": config.get("frozen", True),
        "best_epoch": best_epoch,
        "train_subjects": len(splits["train"]),
        "val_subjects": len(splits["val"]),
        "test_subjects": len(splits["test"]),
        **test_metrics,
    }
    return model, history, pd.Series(result), splits


def run_benchmark(model_names, dataset_names, channel_modes, device=DEVICE):
    rows, histories, models = [], {}, {}
    for dataset_name in dataset_names:
        for model_name in model_names:
            for channel_mode in channel_modes:
                model, history, result, splits = train_probe(
                    model_name, dataset_name, channel_mode, device
                )
                key = (model_name, dataset_name, channel_mode)
                models[key] = model.to("cpu")
                histories[key] = history
                rows.append(result)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    return pd.DataFrame(rows).reset_index(drop=True), histories, models


In [ ]:
# Change these lists to benchmark every requested combination. The head itself
# is selected in utils/configs/<model>.yaml (for example: head: probe).
MODEL_NAMES = [
    # "csbrain",
    "eegpt"
    # "reve"
    ]
DATASET_NAMES = [
    "BCI2A"
    ]
CHANNEL_MODES = [
    "mapped", 
    "raw"
    ]  

results, histories, trained_models = run_benchmark(
    MODEL_NAMES, DATASET_NAMES, CHANNEL_MODES
)
display(results)


[EEGPT] loaded 102/102 keys from /home/infres/yrothlin-24/EEG_Foundation_Model/benchmark/models/checkpoints/EEGPT/eegpt_mcae_58chs_4s_large4E.ckpt
Model=eegpt, head=probe, dataset=BCI2A, channel_mode=mapped, device=cuda
Subjects: train=7, val=1, test=1
backbone params: 25,287,168
classifier params: 22,564
chan_conv params: 1,240
trainable params: 23,804
Epoch 001/050 | train loss=1.3804, bacc=0.2881 | val loss=1.3697, bacc=0.2850
Epoch 002/050 | train loss=1.2871, bacc=0.4163 | val loss=1.3832, bacc=0.3062
Epoch 003/050 | train loss=1.2147, bacc=0.4804 | val loss=1.3936, bacc=0.3155
Epoch 004/050 | train loss=1.1608, bacc=0.5148 | val loss=1.4278, bacc=0.3106
Epoch 005/050 | train loss=1.1262, bacc=0.5415 | val loss=1.4256, bacc=0.3199
Epoch 006/050 | train loss=1.0940, bacc=0.5634 | val loss=1.4042, bacc=0.3068
Epoch 007/050 | train loss=1.0550, bacc=0.5894 | val loss=1.3884, bacc=0.3496
Epoch 008/050 | train loss=1.0298, bacc=0.6034 | val loss=1.4214, bacc=0.3366
Epoch 009/050 | trai